In [1]:
import pandas as pd
import numpy as np
import random
import json
import pickle
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer

In [2]:
p = 0.20

columns = [
    'recommendationid', 'appid', 'game', 'author_steamid', 'author_num_games_owned', 
    'author_num_reviews', 'author_playtime_forever', 'author_playtime_last_two_weeks', 
    'author_playtime_at_review', 'author_last_played', 'voted_up'
]

# Load in processed dataset
PROCESSED__REVIEW_DATASET_PATH = "../data/all_reviews/processed_dataset.csv"

df = pd.read_csv(PROCESSED__REVIEW_DATASET_PATH, header=0, skiprows= lambda i: i>0 and random.random() > p)

# Only keep games with enough data
game_counts = df.groupby('appid')['author_steamid'].count() # Group by appid, then get how many reviews the game has based on steamid
popular_games = game_counts[game_counts > 5].index # Keep only the games that have more than 200 reviews
df = df[df['appid'].isin(popular_games)] # Remake the original df using only appid's that are in popular_games

# Keep only users who play enough games
user_counts = df.groupby('author_steamid')['appid'].count() # Group by steamid, then get how many games theyve reviewed based on appid
active_users = user_counts[user_counts > 5].index # Keep only the users that have reviewed more than 5 games 
df = df[df['author_steamid'].isin(active_users)] # Remake the df using only steamid's that are in active_users

# Force hours columns to be of type numeric
df['author_playtime_forever'] = pd.to_numeric(df['author_playtime_forever'], errors='coerce')

# Convert hours played into a confidence score or rating
df['author_playtime_forever_normalized'] = np.log(df['author_playtime_forever'] + 1)
df['author_playtime_last_two_weeks_normalized'] = np.log(df['author_playtime_last_two_weeks'] + 1)
df['author_playtime_at_review_normalized'] = np.log(df['author_playtime_at_review'] + 1)

# Load in game tag and banned games json files
with open('final_steam_tags_dict.json', 'r') as f:
    final_tags_dict = json.load(f)

with open('final_banned_list.json', 'r') as f:
    true_banned = json.load(f)

with open('final_review_scores.json', 'r') as f:
    review_dict = json.load(f)


In [3]:
# Rebuild the local dictionary from original dataframe
game_mapping = df[['appid', 'game']].drop_duplicates()
name_to_id_local = {str(row['game']).lower().strip(): row['appid'] for _, row in game_mapping.iterrows()}

In [4]:
# Create pivot table
user_game_matrix = df.pivot_table(
    index='author_steamid',
    columns='game',
    values='author_playtime_forever_normalized',
    fill_value=0
)

# Purge NSFW games 
clean_matrix = user_game_matrix.drop(columns=true_banned, errors='ignore')

print(f"Final clean matrix shape: {clean_matrix.shape}")

Final clean matrix shape: (116852, 14378)


In [5]:
# Transpose so that games becomes row, users become columns
SVD = TruncatedSVD(n_components=100, random_state=42)
svd_matrix = SVD.fit_transform(clean_matrix.T)

# Measure angles between the games based on played hours
corr = cosine_similarity(svd_matrix)

In [6]:
# Extract exact column names in order
game_titles = list(clean_matrix.columns)

# Match the text tags to the columns
corpus = []

for game in game_titles:
    tags = final_tags_dict.get(game, "").split()

    if len(tags) >= 3:
        boosted_tags = [tags[0], tags[0], tags[0], tags[1], tags[1]] + tags
        corpus.append(" ".join(boosted_tags))

    else:
        corpus.append(" ".join(tags))

tf_idf = TfidfVectorizer(stop_words="english")
tfidf_matrix = tf_idf.fit_transform(corpus)

# Measure angles between games based on tags
content_corr = cosine_similarity(tfidf_matrix)

In [7]:
# Create the hybrid review/tags correlation
weight_collab = 0.40
weight_content = 0.60

hybrid_corr = (corr * weight_collab) + (content_corr * weight_content)
print("Hybrid engine ready.")

Hybrid engine ready.


#### The function in the cell below already exists in app.py. Uncomment to get recommendations without using the frontend

In [ ]:
# def get_recommendations(game_name, clean_matrix, hybrid_corr, name_to_id, top_n=10):
#     # Create a list of the column names from clean_matrix
#     game_titles = list(clean_matrix.columns)

#     # Check if the game name is in game titles  
#     if game_name not in game_titles:
#         return f"Error: '{game_name}' is not in the matrix"
    
#     # Create the correlation vector
#     idx = game_titles.index(game_name)
#     correlation_vector = hybrid_corr[idx]

#     # Break game_names into pieces, and only keep the core words
#     stop_words = {'the', 'of', 'and', 'in', 'a', 'to', 'for', 'edition', 'remaster'}
#     target_words = set([w for w in game_name.lower().split() if w not in stop_words and not w.isnumeric()])

#     scored_games = []


#     for i, base_score in enumerate(correlation_vector):
#         similar_game_name = game_titles[i]

#         # If the game is the same or similar to game_name, skip it
#         if similar_game_name == game_name:
#             continue

#         sim_words = set([w for w in similar_game_name.lower().split() if w not in stop_words and not w.isnumeric()])
#         if len(target_words.intersection(sim_words)) > 0:
#             continue
        
#         # Calculate review score
#         reviews = review_dict.get(similar_game_name, {'positive': 0, 'negative': 0})
#         pos = reviews['positive']
#         neg = reviews['negative']
#         total = pos + neg

#         quality_ratio = (pos / total) if total > 0 else 0.70

#         final_score = base_score * quality_ratio

#         scored_games.append((similar_game_name, final_score, quality_ratio))

#     scored_games.sort(key=lambda x: x[1], reverse=True)

#     final_result = []

#     for i in range(top_n):
#         game, f_score, q_ratio = scored_games[i]

#         clean_name = str(game).lower().strip()
#         app_id = name_to_id.get(clean_name)

#         final_result.append({
#             "game": clean_name,
#             "app_id": app_id,
#             "match_score": float(f_score),
#             "rating": q_ratio
#         })
    
#     return final_result

# get_recommendations("ELDEN RING", clean_matrix, hybrid_corr, name_to_id_local, top_n=10)

In [ ]:
# Save .pkl files
with open('svd_matrix.pkl', 'wb') as f:
    pickle.dump(svd_matrix, f)

with open('tfidf_matrix.pkl', 'wb') as f:
    pickle.dump(tfidf_matrix, f)

game_titles = list(clean_matrix.columns)
with open('game_titles.pkl', 'wb') as f:
    pickle.dump(game_titles, f)

with open('review_dict.pkl', 'wb') as f:
    pickle.dump(review_dict, f)

with open('name_to_id.pkl', 'wb') as f:
    pickle.dump(name_to_id_local, f)